# KW51 Railway Bridge — Structural Health Monitoring
## Unsupervised Anomaly Detection using Machine Learning

**Dataset:** KW51 Bridge (Maes & Lombaert, KU Leuven) — Zenodo  
**Method:** Environmental correction via Random Forest Regression → Autoencoder anomaly detection  
**Monitoring period:** October 2018 – January 2020 (15 months, hourly resolution)

---
### Pipeline Overview
1. Data loading & initial inspection  
2. Data cleaning (missing values, timestamps, duplicates)  
3. Environmental gap-filling using Open-Meteo historical weather API  
4. Environmental correction — Random Forest Regression  
5. Autoencoder anomaly detection on residuals  
6. Results & visualisation


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import scipy.io

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")


## 2. Data Loading

The KW51 dataset is stored in a MATLAB `.mat` file.  
We use `scipy.io.loadmat` to read it into Python.  
The file contains a structured array called `modes` with fields:  
- `sdn` — timestamps (MATLAB serial date numbers)  
- `f` — natural frequencies (11328 × 14)  
- `xi` — damping ratios  
- `m` — mode shapes  
- `env` — environmental variables (11328 × 11)  
- `labels_m`, `labels_env` — column name labels


In [ ]:
# ── Load the .mat file ──────────────────────────────────────────────────────
# Update this path to match where you saved the KW51 dataset
mat = scipy.io.loadmat(r'YOUR_PATH_TO_KW51.mat', simplify_cells=False)

modes = mat['modes'][0, 0]

sdn  = modes['sdn'].flatten()          # timestamps — MATLAB serial date numbers
f    = modes['f']                       # natural frequencies  (11328 × 14)
env  = modes['env']                     # environmental vars   (11328 × 11)

# Extract column labels
labels_m   = [str(modes['labels_m'][0, i][0])   for i in range(modes['labels_m'].shape[1])]
labels_env = [str(modes['labels_env'][0, i][0]) for i in range(modes['labels_env'].shape[1])]

print("Frequency column labels:", labels_m[:14])
print("Environment column labels:", labels_env)
print(f"Frequency matrix shape: {f.shape}")
print(f"Environment matrix shape: {env.shape}")
print(f"Number of timestamps: {len(sdn)}")


## 3. Build DataFrame

We combine timestamps, frequencies and environmental variables into a single pandas DataFrame.  
Frequency columns are named f1–f14; we'll drop high-missingness ones shortly.


In [ ]:
freq_labels = [f'f{i+1}' for i in range(f.shape[1])]

df_freq = pd.DataFrame(f, columns=freq_labels)
df_env  = pd.DataFrame(env, columns=labels_env)
df_ts   = pd.DataFrame({'timestamp': sdn})

df = pd.concat([df_ts, df_freq, df_env], axis=1)

print(f"DataFrame shape: {df.shape}")
df.head()


## 4. Missing Value Analysis

Before cleaning we quantify how much data is missing per column.  
**Decision rule:** drop any column with > 50 % missing values.  
Columns retained below this threshold will be imputed or forward-filled later.


In [ ]:
def percentageNAN(df):
    percentage = (df.isnull().sum() / df.shape[0]) * 100
    return percentage

nan_pct = percentageNAN(df)
print("NaN percentage per column:")
print(nan_pct.round(2).to_string())


### 4.1 Drop columns with > 50 % missing

`f7` (~70 %) and `f8` (~72 %) exceed the threshold and are dropped.


In [ ]:
to_drop = nan_pct[nan_pct > 50].index.tolist()
print("Columns to drop (> 50% NaN):", to_drop)

df = df.drop(columns=to_drop, axis=1)
print(f"DataFrame shape after dropping: {df.shape}")


## 5. Timestamp Conversion

The timestamps are MATLAB serial date numbers — days since January 1, year 0.  
Python/pandas counts from January 1, 1970 (Unix epoch).  
The offset between the two epochs is **719 529 days**.

We subtract this offset then use `pd.to_datetime(unit='D')` to convert.


In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'] - 719529, unit='D')
print("First timestamp:", df['timestamp'].iloc[0])
print("Last timestamp: ", df['timestamp'].iloc[-1])


### 5.1 Set timestamp as index & round to nearest hour

We set the timestamp as the DataFrame index — this enables clean date-based slicing.  
We also round to the nearest hour so timestamps align with the hourly weather API data we fetch later.


In [ ]:
df = df.set_index('timestamp')
df.index = df.index.round('H')

print("Index dtype:", df.index.dtype)
print("Duplicate timestamps:", df.index.duplicated().sum())
print(df.index[:3])


## 6. Data Type & Duplicate Checks

In [ ]:
print("Column dtypes:")
print(df.dtypes)
print()
print(f"Duplicate rows: {df.duplicated().sum()}")


## 7. Visual Inspection of Frequency Columns

Plotting all frequency columns over time to check for flat/dead sensors and to observe the structural events (retrofitting ~mid-2019).


In [ ]:
freq_cols_available = [c for c in df.columns if c.startswith('f')]

fig, axes = plt.subplots(len(freq_cols_available), 1, figsize=(15, 2.5 * len(freq_cols_available)))

for i, col in enumerate(freq_cols_available):
    axes[i].plot(df.index, df[col], linewidth=0.5, color='steelblue')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel('Hz', fontsize=8)

plt.tight_layout()
plt.savefig('frequency_overview.png', dpi=120, bbox_inches='tight')
plt.show()


## 8. Drop Additional Columns

**f4** has a ~93-day consecutive gap (2230 rows) — too large to impute reliably.  
**grVL, drVL, dnrVL** (radiation variables) have no direct external replacement and are dropped.


In [ ]:
cols_to_drop = ['f4', 'grVL', 'drVL', 'dnrVL']
# Only drop if they exist (in case this cell is re-run)
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

df = df.drop(columns=cols_to_drop)
print(f"Dropped: {cols_to_drop}")
print(f"DataFrame shape: {df.shape}")


## 9. Fill Environmental Gaps with Open-Meteo Historical Weather

Several environmental columns have multi-week gaps that forward-fill alone cannot handle.  
We fetch hourly historical weather data for Leuven, Belgium from the Open-Meteo archive API — no API key required.

**Mapped columns:**  
| Bridge column | Open-Meteo variable |
|---|---|
| tBD31A, tVL | temperature_2m |
| rhBD31A, rhVL | relative_humidity_2m |
| vpVL | vapour_pressure_deficit |
| raVL | rain |
| wsVL | windspeed_10m |
| wdVL | winddirection_10m |


In [ ]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 50.8798,
    "longitude": 4.7005,
    "start_date": "2018-10-01",
    "end_date": "2020-01-31",
    "hourly": "temperature_2m,relative_humidity_2m,vapour_pressure_deficit,rain,windspeed_10m,winddirection_10m",
    "timezone": "Europe/Brussels"
}

response = requests.get(url, params=params)
weather_raw = response.json()

weather_df = pd.DataFrame(weather_raw['hourly'])
weather_df['time'] = pd.to_datetime(weather_df['time'])
weather_df = weather_df.set_index('time')
weather_df.index.name = 'timestamp'

print(f"Weather DataFrame shape: {weather_df.shape}")
print(weather_df.head(3))


In [ ]:
# Fill gaps in bridge environmental columns using weather data
fill_map = {
    'tBD31A':  'temperature_2m',
    'rhBD31A': 'relative_humidity_2m',
    'tVL':     'temperature_2m',
    'rhVL':    'relative_humidity_2m',
    'vpVL':    'vapour_pressure_deficit',
    'raVL':    'rain',
    'wsVL':    'windspeed_10m',
    'wdVL':    'winddirection_10m',
}

for bridge_col, weather_col in fill_map.items():
    if bridge_col in df.columns:
        before = df[bridge_col].isnull().sum()
        df[bridge_col] = df[bridge_col].fillna(weather_df[weather_col])
        after = df[bridge_col].isnull().sum()
        print(f"{bridge_col}: {before} NaN → {after} NaN")


## 10. Forward Fill Remaining Missing Values

For frequency columns with short gaps (< 70 consecutive rows), we use forward fill — propagating the last valid reading forward.  
For any NaNs at the very start of the series (where forward fill cannot reach), we apply backward fill.


In [ ]:
df = df.ffill()
df = df.bfill()

print(f"Total NaNs remaining: {df.isnull().sum().sum()}")
print(f"Final DataFrame shape: {df.shape}")
df.info()


## 11. Save Cleaned Data

In [ ]:
# Update path as needed
df.to_csv(r'kw51_cleaned.csv')
print("Cleaned data saved.")


## 12. Environmental Correction — Random Forest Regression

Natural frequencies are strongly influenced by temperature and humidity.  
A frequency drop in winter does not necessarily indicate structural damage — it may simply reflect thermal contraction.

**Goal:** separate the environmentally-driven frequency variation from true structural changes.

**Method:**  
1. Train one Random Forest Regressor per frequency column, using environmental variables as inputs  
2. Train only on the **pre-damage period** (Oct 2018 – Apr 2019) so the model learns *normal* behaviour  
3. Predict expected frequencies for the **entire dataset**  
4. Compute residuals: `residual = actual − predicted`

The residuals represent frequency deviations that cannot be explained by environmental conditions alone.  
This is the signal we feed to the anomaly detector.


In [ ]:
freq_cols = ['f3', 'f5', 'f6', 'f9', 'f10', 'f11', 'f12', 'f13']
env_cols  = ['tBD31A', 'rhBD31A', 'tVL', 'rhVL', 'vpVL', 'raVL', 'wsVL', 'wdVL']

# Training slice — pre-damage normal period
x_train = df[env_cols].loc['2018-10-01':'2019-04-30']
y_train = df[freq_cols].loc['2018-10-01':'2019-04-30']

# Train one model per frequency mode
models = {}
for col in freq_cols:
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(x_train, y_train[col])
    models[col] = rf
    print(f"Trained model for {col}")

print("\nAll models trained.")


In [ ]:
# Predict on entire dataset
predictions = {}
for col in freq_cols:
    predictions[col] = models[col].predict(df[env_cols])

pred_df = pd.DataFrame(predictions, index=df.index)

# Residuals: actual minus environmentally-predicted
residuals_df = df[freq_cols] - pred_df

print(f"Residuals shape: {residuals_df.shape}")
print("\nResidual statistics (training period):")
print(residuals_df.loc['2018-10-01':'2019-04-30'].describe().round(4))


In [ ]:
# Visual check — residuals for f9 over full period
# Should be near-zero in training period, large deviation in damage period
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(residuals_df.index, residuals_df['f9'], linewidth=0.5, color='steelblue')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvspan(pd.Timestamp('2018-10-01'), pd.Timestamp('2019-04-30'),
           alpha=0.08, color='green', label='Training period')
ax.set_title('f9 residuals over full monitoring period')
ax.set_ylabel('Residual (Hz)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('f9_residuals.png', dpi=120, bbox_inches='tight')
plt.show()


## 13. Autoencoder Anomaly Detection

An autoencoder is a neural network trained to **compress and reconstruct** its input.  
When trained only on normal data, it learns what normal residual patterns look like.  

**Anomaly detection mechanism:**  
- Normal data → low reconstruction error (the model has seen this before)  
- Anomalous data → high reconstruction error (the pattern is unfamiliar)

**Architecture:** 8 → 6 → 3 → 6 → 8 (dense layers, ReLU activations)

**Preprocessing:**  
- Clip residuals to ±3 standard deviations (computed from training period) to reduce the influence of isolated sensor spikes  
- Scale to zero mean, unit variance using `StandardScaler` fit on training data only


In [ ]:
# ── Clip residuals at ±3 std (training period stats only) ───────────────────
train_period = residuals_df.loc['2018-10-01':'2019-04-30']
train_mean   = train_period.mean()
train_std    = train_period.std()

lower = train_mean - 3 * train_std
upper = train_mean + 3 * train_std

residuals_clipped = residuals_df.clip(lower=lower, upper=upper, axis=1)

# ── Prepare train / full arrays ──────────────────────────────────────────────
X_full  = residuals_clipped.values
X_train = residuals_clipped.loc['2018-10-01':'2019-04-30'].values

# ── Scale ────────────────────────────────────────────────────────────────────
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train)   # fit on normal period only
X_full_sc    = scaler.transform(X_full)         # apply same scaling to full data

print(f"Training samples : {X_train_sc.shape[0]}")
print(f"Full samples     : {X_full_sc.shape[0]}")


In [ ]:
# ── Build autoencoder ────────────────────────────────────────────────────────
model = Sequential([
    Dense(6, activation='relu', input_shape=(8,)),   # encoder
    Dense(3, activation='relu'),                      # bottleneck
    Dense(6, activation='relu'),                      # decoder
    Dense(8, activation='linear')                     # reconstruction
])

model.compile(optimizer='adam', loss='mse')
model.summary()


In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
# Input and target are the same — the model learns to reconstruct its input
history = model.fit(
    X_train_sc, X_train_sc,
    epochs=100,
    batch_size=32,
    verbose=1
)

print(f"\nFinal training loss: {history.history['loss'][-1]:.4f}")


In [ ]:
# ── Training loss curve ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(history.history['loss'], color='steelblue', linewidth=1)
ax.set_title('Autoencoder training loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
plt.tight_layout()
plt.savefig('training_loss.png', dpi=120, bbox_inches='tight')
plt.show()


## 14. Reconstruction Error & Anomaly Threshold

For each time point we compute the **mean squared error** between the actual residual and its reconstruction.  
The anomaly threshold is set at: **mean + 3 × std** of the training-period reconstruction error.  
This follows the statistical convention that values beyond 3 standard deviations are genuinely unusual.


In [ ]:
# ── Reconstruct & compute per-sample MSE ─────────────────────────────────────
reconstructed = model.predict(X_full_sc, verbose=0)
mse           = np.mean((X_full_sc - reconstructed) ** 2, axis=1)
mse_series    = pd.Series(mse, index=residuals_clipped.index)

# ── Threshold from training period ───────────────────────────────────────────
train_mse = mse_series['2018-10-01':'2019-04-30']
threshold = train_mse.mean() + 3 * train_mse.std()

anomalies = mse_series[mse_series > threshold]

print(f"Anomaly threshold  : {threshold:.4f}")
print(f"Training MSE mean  : {train_mse.mean():.4f}")
print(f"Training MSE std   : {train_mse.std():.4f}")
print(f"Total flagged      : {len(anomalies):,} / {len(mse_series):,} ({len(anomalies)/len(mse_series)*100:.1f}%)")


## 15. Results

The plot below shows the reconstruction error over the full 15-month monitoring period.  
- **Blue line** — MSE at each time point  
- **Red dashed line** — anomaly threshold (mean + 3σ from training period)  
- **Red shading** — flagged anomaly regions  
- **Orange band** — approximate retrofitting period (May–September 2019)

The model correctly identifies a sustained anomaly corresponding to the known bridge retrofitting event.  
The post-repair period also shows elevated error because the bridge settled at a slightly different frequency baseline — consistent with a physical change in structural state.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(mse_series.index, mse_series.values,
        linewidth=0.5, color='steelblue', alpha=0.85, label='Reconstruction error (MSE)')

ax.axhline(y=threshold, color='crimson', linewidth=1.5,
           linestyle='--', label=f'Anomaly threshold ({threshold:.2f})')

ax.axvspan(pd.Timestamp('2019-05-01'), pd.Timestamp('2019-09-30'),
           alpha=0.12, color='orange', label='Retrofitting period (approx.)')

ax.fill_between(mse_series.index, mse_series.values, threshold,
                where=(mse_series.values > threshold),
                color='crimson', alpha=0.2, label='Flagged anomalies')

ax.set_title(
    'KW51 Railway Bridge — Anomaly Detection\n'
    'Autoencoder Reconstruction Error | Environmental Correction Applied',
    fontsize=12
)
ax.set_xlabel('Date')
ax.set_ylabel('Reconstruction Error (MSE)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.savefig('anomaly_detection_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("Final plot saved as anomaly_detection_result.png")


## 16. Project Summary

In [ ]:
total   = len(mse_series)
flagged = (mse_series > threshold).sum()

print("=" * 50)
print("  KW51 BRIDGE — ANOMALY DETECTION SUMMARY")
print("=" * 50)
print(f"  Dataset         : KW51 Railway Bridge, Leuven")
print(f"  Monitoring      : Oct 2018 – Jan 2020")
print(f"  Time points     : {total:,} (hourly)")
print(f"  Frequency modes : 8  (f3, f5, f6, f9–f13)")
print(f"  Env. variables  : 8")
print()
print(f"  Training period : Oct 2018 – Apr 2019")
print(f"  Env. correction : Random Forest Regression")
print(f"  Anomaly model   : Dense Autoencoder (8→6→3→6→8)")
print()
print(f"  Threshold       : {threshold:.4f}  (mean + 3σ)")
print(f"  Flagged points  : {flagged:,} / {total:,}  ({flagged/total*100:.1f}%)")
print()
print(f"  Key finding     : Sustained anomaly detected from")
print(f"                    ~May 2019, consistent with known")
print(f"                    bridge retrofitting event.")
print("=" * 50)
